In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import os, csv, json, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter, defaultdict

In [4]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
 
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")

Device: cuda


In [5]:
# Load annotations with scene type
annotations = {}
with open(os.path.join(ZIP1, "annotation.csv"), 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            cid = row[0].strip()
            annotations[cid] = {
                'text': row[1].strip() if len(row) > 1 else '',
                'scene_type': row[4].strip() if len(row) > 4 and row[4].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'uncertainty': row[8].strip() if len(row) > 8 and row[8].strip() else None,
            }

In [6]:
# Build index
samples = []
with open(os.path.join(ZIP1, "sample.csv"), 'r') as f:
    reader = csv.reader(f); next(reader)
    for row in reader: samples.append(row)
 
emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5,'surprise':6}
polarity_map = {'positive':0,'neutral':1,'negative':2}
intensity_map = {'weak':0,'powerful':1}
emo_names = ['angry','disgust','fear','happy','neutral','sad','surprise']

In [7]:
# Discover scene types
scene_types = set()
for ann in annotations.values():
    if ann.get('scene_type'): scene_types.add(ann['scene_type'])
scene_types = sorted(scene_types)
scene_map = {s: i for i, s in enumerate(scene_types)}
print(f"Scene types: {scene_types}")
print(f"Scene map: {scene_map}")
 
mcis_index = []
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {'sample_id': s[0].strip(), 'clip_ids': clips,
             'feature_files': [c.replace('/','_')+'.pt' for c in clips]}
    
    for ln, ci in [('clip3',2),('clip4',3)]:
        cid = clips[ci]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{ln}_emotion'] = emotion_map.get(ann['emotion'],-1)
            entry[f'{ln}_polarity'] = polarity_map.get(ann.get('polarity',''),-1)
            entry[f'{ln}_intensity'] = intensity_map.get(ann.get('intensity',''),-1)
            entry[f'{ln}_uncertainty'] = int(ann['uncertainty']) if ann.get('uncertainty','').isdigit() else -1
            entry[f'{ln}_scene'] = scene_map.get(ann.get('scene_type',''),-1)
        else:
            entry[f'{ln}_emotion'] = -1; entry[f'{ln}_polarity'] = -1
            entry[f'{ln}_intensity'] = -1; entry[f'{ln}_uncertainty'] = -1
            entry[f'{ln}_scene'] = -1
    mcis_index.append(entry)

Scene types: ['daily', 'entertainment', 'social', 'work']
Scene map: {'daily': 0, 'entertainment': 1, 'social': 2, 'work': 3}


In [8]:
# Split
def split_mcis(mcis_index, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    clip4_groups = {}
    for idx, e in enumerate(mcis_index):
        c4 = e['clip_ids'][3]
        clip4_groups.setdefault(c4, []).append(idx)
    keys = list(clip4_groups.keys()); rng.shuffle(keys)
    n = len(keys); nt = int(n*train_ratio); nv = int(n*val_ratio)
    return ([i for g in keys[:nt] for i in clip4_groups[g]],
            [i for g in keys[nt:nt+nv] for i in clip4_groups[g]],
            [i for g in keys[nt+nv:] for i in clip4_groups[g]])
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Train: 1980, Val: 424, Test: 426


In [9]:
# %%
class HiEFDataset(Dataset):
    def __init__(self, mcis_index, features_dir, indices):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
    def __len__(self): return len(self.indices)
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        c3f,c3o,c3t,c3a = self._load(e['feature_files'][2])
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            'clip3_face':c3f,'clip3_ori':c3o,'clip3_text':c3t,'clip3_audio':c3a,
            'target':e['clip4_emotion'],
            'clip3_emotion':e['clip3_emotion'],
            'clip3_scene':e.get('clip3_scene',-1),
        }
 
def collate_fn(batch):
    r = {}
    for k in [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]:
        r[k] = torch.stack([b[k] for b in batch])
    for k in ['target','clip3_emotion','clip3_scene']:
        r[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
    return r
 
BATCH_SIZE = 32
train_dataset = HiEFDataset(mcis_index, FEATURES_DIR, train_idx)
val_dataset = HiEFDataset(mcis_index, FEATURES_DIR, val_idx)
test_dataset = HiEFDataset(mcis_index, FEATURES_DIR, test_idx)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
print("Data loaded ✓")

Data loaded ✓


In [10]:
# %%
class TemporalTransformer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)
        el = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
    def forward(self, x):
        return self.transformer(x + self.pos_encoding[:, :x.size(1), :])
 
class CrossAttentionFusion(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_layers=1, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
    def forward(self, query, kv):
        x = query
        for attn, norm in zip(self.layers, self.norms):
            out, _ = attn(x, kv, kv)
            x = norm(x + out)
        return x

In [11]:
class IntraEncoder(nn.Module):
    """Configurable intra-video encoder.
    query_modality controls which modality is the query in modality fusion:
      'face' = face-driven (baseline behavior, good for emotion)
      'text' = text-driven (good for context/topic understanding)
    """
    def __init__(self, d_model=512, audio_dim=527, query_modality='face'):
        super().__init__()
        self.query_modality = query_modality
        self.face_temporal = TemporalTransformer(d_model)
        self.ori_temporal = TemporalTransformer(d_model)
        self.type_fusion = CrossAttentionFusion(d_model)
        self.audio_proj = nn.Linear(audio_dim, d_model)
        self.modality_fusion = CrossAttentionFusion(d_model)
    
    def forward(self, face, ori, text, audio):
        face_out = self.face_temporal(face).mean(dim=1, keepdim=True)
        ori_out = self.ori_temporal(ori).mean(dim=1, keepdim=True)
        
        vis_stack = torch.cat([face_out, ori_out], dim=1)
        video_feat = self.type_fusion(face_out, vis_stack)  # [B,1,512]
        
        audio_feat = self.audio_proj(F.normalize(audio, dim=-1)).unsqueeze(1)
        text_feat = text.unsqueeze(1)
        
        mod_stack = torch.cat([video_feat, text_feat, audio_feat], dim=1)  # [B,3,512]
        
        if self.query_modality == 'text':
            query = text_feat    # context: text drives attention
        else:
            query = video_feat   # emotion: face/video drives attention
        
        clip_feat = self.modality_fusion(query, mod_stack)
        return clip_feat.squeeze(1)  # [B, 512]

In [12]:
# %%
class Baseline(nn.Module):
    """Original: shared encoder, LSTM+Transformer inter-fusion."""
    def __init__(self, d=512, n_classes=7):
        super().__init__()
        self.encoder = IntraEncoder(d, query_modality='face')
        self.lstm = nn.LSTM(d, d, num_layers=3, batch_first=False, dropout=0.1)
        self.pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, 8, d*4, 0.1, batch_first=True, norm_first=True)
        self.trans = nn.TransformerEncoder(el, num_layers=2)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
    
    def forward(self, batch):
        f1 = self.encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        f2 = self.encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        f3 = self.encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        seq = torch.stack([f1,f2,f3], dim=0)
        out, _ = self.lstm(seq)
        out = self.trans(out.permute(1,0,2) + self.pos).mean(dim=1)
        return self.head(out), {}
 
 
class Gap1v1(nn.Module):
    """Level 1: Separate weights. Option B inter-fusion."""
    def __init__(self, d=512, n_classes=7):
        super().__init__()
        self.ctx_encoder = IntraEncoder(d, query_modality='face')   # same arch, separate weights
        self.emo_encoder = IntraEncoder(d, query_modality='face')   # same arch, separate weights
        
        # Context fusion: merge clip I + II into one context vector
        self.ctx_fusion = CrossAttentionFusion(d, n_heads=8, n_layers=1)
        
        # Forecasting: cross-attend emotion with context
        self.forecast = CrossAttentionFusion(d, n_heads=8, n_layers=2)
        
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
    
    def forward(self, batch):
        ctx1 = self.ctx_encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        ctx2 = self.ctx_encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        emo3 = self.emo_encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        
        # f_CI: fuse context clips
        ctx_stack = torch.stack([ctx1, ctx2], dim=1)          # [B, 2, 512]
        ctx_combined = self.ctx_fusion(ctx1.unsqueeze(1), ctx_stack).squeeze(1)  # [B, 512]
        
        # f_EF: emotion attends to [emotion, context]
        emo_q = emo3.unsqueeze(1)                             # [B, 1, 512]
        ef_stack = torch.cat([emo_q, ctx_combined.unsqueeze(1)], dim=1)  # [B, 2, 512]
        final = self.forecast(emo_q, ef_stack).squeeze(1)     # [B, 512]
        
        return self.head(final), {}
 
 
class Gap1v2(nn.Module):
    """Level 2: + Different modality emphasis."""
    def __init__(self, d=512, n_classes=7):
        super().__init__()
        self.ctx_encoder = IntraEncoder(d, query_modality='text')   # text-driven for context
        self.emo_encoder = IntraEncoder(d, query_modality='face')   # face-driven for emotion
        
        self.ctx_fusion = CrossAttentionFusion(d, n_heads=8, n_layers=1)
        self.forecast = CrossAttentionFusion(d, n_heads=8, n_layers=2)
        
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
    
    def forward(self, batch):
        ctx1 = self.ctx_encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        ctx2 = self.ctx_encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        emo3 = self.emo_encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        
        ctx_stack = torch.stack([ctx1, ctx2], dim=1)
        ctx_combined = self.ctx_fusion(ctx1.unsqueeze(1), ctx_stack).squeeze(1)
        
        emo_q = emo3.unsqueeze(1)
        ef_stack = torch.cat([emo_q, ctx_combined.unsqueeze(1)], dim=1)
        final = self.forecast(emo_q, ef_stack).squeeze(1)
        
        return self.head(final), {}
 
 
class Gap1v3(nn.Module):
    """Level 3: + Auxiliary losses forcing disentanglement."""
    def __init__(self, d=512, n_classes=7, n_scenes=4):
        super().__init__()
        self.ctx_encoder = IntraEncoder(d, query_modality='text')
        self.emo_encoder = IntraEncoder(d, query_modality='face')
        
        self.ctx_fusion = CrossAttentionFusion(d, n_heads=8, n_layers=1)
        self.forecast = CrossAttentionFusion(d, n_heads=8, n_layers=2)
        
        # Main head: predict B's emotion
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))
        
        # Auxiliary head 1: context encoder should predict scene type
        self.scene_head = nn.Linear(d, n_scenes)
        
        # Auxiliary head 2: emotion encoder should predict A's emotion
        self.emo_aux_head = nn.Linear(d, n_classes)
    
    def forward(self, batch):
        ctx1 = self.ctx_encoder(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        ctx2 = self.ctx_encoder(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        emo3 = self.emo_encoder(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        
        ctx_stack = torch.stack([ctx1, ctx2], dim=1)
        ctx_combined = self.ctx_fusion(ctx1.unsqueeze(1), ctx_stack).squeeze(1)
        
        emo_q = emo3.unsqueeze(1)
        ef_stack = torch.cat([emo_q, ctx_combined.unsqueeze(1)], dim=1)
        final = self.forecast(emo_q, ef_stack).squeeze(1)
        
        aux = {
            'scene_logits': self.scene_head(ctx_combined),  # [B, n_scenes]
            'emo_a_logits': self.emo_aux_head(emo3),        # [B, 7]
        }
        
        return self.head(final), aux

In [13]:
# %%
def compute_metrics(preds, labels, n_classes=7):
    preds, labels = np.array(preds), np.array(labels)
    war = (preds == labels).sum() / len(labels) * 100
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0: recalls.append((preds[mask] == c).sum() / mask.sum() * 100)
    return war, np.mean(recalls) if recalls else 0.0
 
def train_model(model, train_loader, val_loader, test_loader, n_epochs=50, lr=1e-4, 
                aux_weight=0.3, model_name="model", device=DEVICE):
    """Train with optional auxiliary losses."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_uar = 0
    best_epoch = 0
    
    for epoch in range(1, n_epochs + 1):
        # --- Train ---
        model.train()
        for batch in train_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            logits, aux = model(bg)
            
            loss = F.cross_entropy(logits, bg['target'])
            
            # Auxiliary losses (Gap1-v3)
            if 'scene_logits' in aux and (bg['clip3_scene'] >= 0).any():
                mask = bg['clip3_scene'] >= 0
                if mask.sum() > 0:
                    loss += aux_weight * F.cross_entropy(aux['scene_logits'][mask], bg['clip3_scene'][mask])
            
            if 'emo_a_logits' in aux and (bg['clip3_emotion'] >= 0).any():
                mask = bg['clip3_emotion'] >= 0
                if mask.sum() > 0:
                    loss += aux_weight * F.cross_entropy(aux['emo_a_logits'][mask], bg['clip3_emotion'][mask])
            
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # --- Val ---
        model.eval(); vp, vl = [], []; val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                bg = {k: v.to(device) for k, v in batch.items()}
                logits, _ = model(bg)
                val_loss += F.cross_entropy(logits, bg['target']).item() * logits.size(0)
                vp.extend(logits.argmax(-1).cpu().numpy())
                vl.extend(batch['target'].numpy())
        
        scheduler.step(val_loss / len(val_loader.dataset))
        vw, vu = compute_metrics(vp, vl)
        
        if vu > best_val_uar:
            best_val_uar = vu
            best_epoch = epoch
            torch.save(model.state_dict(), f'/kaggle/working/{model_name}_best.pt')
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{model_name}] Ep {epoch:>3}: Val WAR={vw:.1f}%, UAR={vu:.1f}%"
                  f"{'  ★' if epoch == best_epoch else ''}")
    
    # --- Test with best model ---
    model.load_state_dict(torch.load(f'/kaggle/working/{model_name}_best.pt', map_location=device, weights_only=True))
    model.eval(); tp, tl = [], []
    with torch.no_grad():
        for batch in test_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            logits, _ = model(bg)
            tp.extend(logits.argmax(-1).cpu().numpy())
            tl.extend(batch['target'].numpy())
    
    tw, tu = compute_metrics(tp, tl)
    
    # Per-class recall
    tp_arr, tl_arr = np.array(tp), np.array(tl)
    per_class = {}
    for c in range(7):
        mask = tl_arr == c
        if mask.sum() > 0:
            per_class[emo_names[c]] = (tp_arr[mask] == c).sum() / mask.sum() * 100
        else:
            per_class[emo_names[c]] = 0.0
    
    # Count classes never predicted
    predicted_classes = len(set(tp_arr))
    
    return {
        'model_name': model_name,
        'best_epoch': best_epoch,
        'best_val_uar': best_val_uar,
        'test_war': tw,
        'test_uar': tu,
        'per_class': per_class,
        'n_predicted_classes': predicted_classes,
        'test_preds': tp_arr,
        'test_labels': tl_arr,
    }

In [14]:
print("=" * 60)
print("Training 4 models (50 epochs each)")
print("=" * 60)
 
results = {}
 
# --- Baseline ---
print("\n[1/4] Baseline (shared encoder, LSTM+Transformer)")
m = Baseline().to(DEVICE)
params = sum(p.numel() for p in m.parameters())
print(f"  Parameters: {params:,}")
results['baseline'] = train_model(m, train_loader, val_loader, test_loader, model_name="baseline")
 
# --- Gap1-v1 ---
print("\n[2/4] Gap1-v1 (separate weights, Option B fusion)")
m = Gap1v1().to(DEVICE)
params = sum(p.numel() for p in m.parameters())
print(f"  Parameters: {params:,}")
results['gap1_v1'] = train_model(m, train_loader, val_loader, test_loader, model_name="gap1_v1")
 
# --- Gap1-v2 ---
print("\n[3/4] Gap1-v2 (+ text-query for context, face-query for emotion)")
m = Gap1v2().to(DEVICE)
params = sum(p.numel() for p in m.parameters())
print(f"  Parameters: {params:,}")
results['gap1_v2'] = train_model(m, train_loader, val_loader, test_loader, model_name="gap1_v2")
 
# --- Gap1-v3 ---
print("\n[4/4] Gap1-v3 (+ auxiliary losses: scene type + A's emotion)")
m = Gap1v3(n_scenes=len(scene_map)).to(DEVICE)
params = sum(p.numel() for p in m.parameters())
print(f"  Parameters: {params:,}")
results['gap1_v3'] = train_model(m, train_loader, val_loader, test_loader, model_name="gap1_v3", aux_weight=0.3)

Training 4 models (50 epochs each)

[1/4] Baseline (shared encoder, LSTM+Transformer)


/tmp/ipykernel_58/2139794074.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
/tmp/ipykernel_58/1312887224.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  Parameters: 27,743,751
  [baseline] Ep   1: Val WAR=24.1%, UAR=14.3%  ★
  [baseline] Ep  10: Val WAR=39.2%, UAR=25.8%  ★
  [baseline] Ep  20: Val WAR=37.5%, UAR=26.0%
  [baseline] Ep  30: Val WAR=33.3%, UAR=24.3%
  [baseline] Ep  40: Val WAR=31.6%, UAR=23.3%
  [baseline] Ep  50: Val WAR=31.4%, UAR=23.6%

[2/4] Gap1-v1 (separate weights, Option B fusion)
  Parameters: 33,288,199


/tmp/ipykernel_58/2139794074.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)


  [gap1_v1] Ep   1: Val WAR=25.0%, UAR=15.4%  ★
  [gap1_v1] Ep  10: Val WAR=32.8%, UAR=23.8%
  [gap1_v1] Ep  20: Val WAR=33.0%, UAR=25.1%
  [gap1_v1] Ep  30: Val WAR=32.5%, UAR=23.5%
  [gap1_v1] Ep  40: Val WAR=32.1%, UAR=23.1%
  [gap1_v1] Ep  50: Val WAR=32.1%, UAR=22.7%

[3/4] Gap1-v2 (+ text-query for context, face-query for emotion)
  Parameters: 33,288,199
  [gap1_v2] Ep   1: Val WAR=23.8%, UAR=14.7%  ★
  [gap1_v2] Ep  10: Val WAR=35.1%, UAR=24.8%
  [gap1_v2] Ep  20: Val WAR=32.1%, UAR=24.3%
  [gap1_v2] Ep  30: Val WAR=33.0%, UAR=24.8%
  [gap1_v2] Ep  40: Val WAR=32.1%, UAR=23.8%
  [gap1_v2] Ep  50: Val WAR=32.1%, UAR=24.0%

[4/4] Gap1-v3 (+ auxiliary losses: scene type + A's emotion)
  Parameters: 33,293,842
  [gap1_v3] Ep   1: Val WAR=23.1%, UAR=14.3%  ★
  [gap1_v3] Ep  10: Val WAR=36.8%, UAR=26.7%
  [gap1_v3] Ep  20: Val WAR=36.1%, UAR=26.9%
  [gap1_v3] Ep  30: Val WAR=36.1%, UAR=27.0%
  [gap1_v3] Ep  40: Val WAR=35.1%, UAR=26.3%
  [gap1_v3] Ep  50: Val WAR=35.1%, UAR=26.3%


In [15]:
print("\n" + "=" * 80)
print("RESULTS COMPARISON")
print("=" * 80)
 
print(f"\n{'Model':<20} | {'Params':>10} | {'Best Ep':>7} | {'Val UAR':>7} | {'Test WAR':>8} | {'Test UAR':>8} | {'Classes':>7}")
print("-" * 85)
 
param_counts = {
    'baseline': sum(p.numel() for p in Baseline().parameters()),
    'gap1_v1': sum(p.numel() for p in Gap1v1().parameters()),
    'gap1_v2': sum(p.numel() for p in Gap1v2().parameters()),
    'gap1_v3': sum(p.numel() for p in Gap1v3(n_scenes=len(scene_map)).parameters()),
}
 
for name in ['baseline', 'gap1_v1', 'gap1_v2', 'gap1_v3']:
    r = results[name]
    pc = param_counts[name]
    print(f"{r['model_name']:<20} | {pc:>10,} | {r['best_epoch']:>7} | {r['best_val_uar']:>6.1f}% | "
          f"{r['test_war']:>7.1f}% | {r['test_uar']:>7.1f}% | {r['n_predicted_classes']:>3}/7")
 
# Per-class recall comparison
print(f"\n{'Model':<20} |", end="")
for e in emo_names:
    print(f" {e[:7]:>7}", end="")
print()
print("-" * 76)
 
for name in ['baseline', 'gap1_v1', 'gap1_v2', 'gap1_v3']:
    r = results[name]
    print(f"{r['model_name']:<20} |", end="")
    for e in emo_names:
        val = r['per_class'].get(e, 0)
        marker = " ✗" if val == 0 else ""
        print(f" {val:>5.1f}%{marker}" if val > 0 else f"   0.0% ✗", end="")
    print()


RESULTS COMPARISON

Model                |     Params | Best Ep | Val UAR | Test WAR | Test UAR | Classes
-------------------------------------------------------------------------------------


/tmp/ipykernel_58/2139794074.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(el, num_layers=n_layers)
/tmp/ipykernel_58/1312887224.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


baseline             | 27,743,751 |      19 |   27.2% |    37.1% |    25.7% |   5/7
gap1_v1              | 33,288,199 |      25 |   29.4% |    30.5% |    22.3% |   7/7
gap1_v2              | 33,288,199 |      24 |   28.1% |    27.9% |    21.0% |   7/7
gap1_v3              | 33,293,842 |       7 |   28.2% |    37.6% |    26.1% |   6/7

Model                |   angry disgust    fear   happy neutral     sad surpris
----------------------------------------------------------------------------
baseline             |  34.2%   0.0% ✗   0.0% ✗  53.5%  53.7%  32.8%   5.6%
gap1_v1              |  27.6%  17.8%   0.0% ✗  35.4%  47.2%  22.4%   5.6%
gap1_v2              |  26.3%  13.3%   0.0% ✗  39.4%  34.3%  22.4%  11.1%
gap1_v3              |  27.6%  15.6%   0.0% ✗  56.6%  55.6%  27.6%   0.0% ✗


In [16]:
print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
 
baseline_uar = results['baseline']['test_uar']
for name in ['gap1_v1', 'gap1_v2', 'gap1_v3']:
    r = results[name]
    delta = r['test_uar'] - baseline_uar
    direction = "↑" if delta > 0 else "↓"
    print(f"\n{r['model_name']}:")
    print(f"  UAR change vs baseline: {direction} {abs(delta):.1f} points")
    
    # Check if fear/surprise are now predicted
    fear_recall = r['per_class'].get('fear', 0)
    surprise_recall = r['per_class'].get('surprise', 0)
    if fear_recall > 0 or surprise_recall > 0:
        print(f"  ✓ Now predicts fear ({fear_recall:.1f}%) and/or surprise ({surprise_recall:.1f}%)")
    else:
        print(f"  ✗ Still never predicts fear or surprise")


KEY TAKEAWAYS

gap1_v1:
  UAR change vs baseline: ↓ 3.4 points
  ✓ Now predicts fear (0.0%) and/or surprise (5.6%)

gap1_v2:
  UAR change vs baseline: ↓ 4.7 points
  ✓ Now predicts fear (0.0%) and/or surprise (11.1%)

gap1_v3:
  UAR change vs baseline: ↑ 0.4 points
  ✗ Still never predicts fear or surprise
